In [9]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv1D, Dense, Flatten, Dropout, Input, MaxPooling1D, BatchNormalization
warnings.filterwarnings('ignore')
import yfinance as yf

curr = os.getcwd()
sys.path.append(os.path.abspath(os.path.join(curr, '..')))
from features import FEATURES
from help_folder.backtest import backtest, production_backtest, filter_signals
from help_folder.help import prepare_data,calculate_base_features, save_models, generate_final_report, download, optimize_threshold, stage1, stage2, save_artifacts_json, print_stage_report

In [ ]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

TICKER = "AAPL"                  
START_DATE = "2010-01-01"       
END_DATE = "2023-12-31"          
INTERVAL = "1d"        

TRAIN_SPLIT = 0.70               
VAL_SPLIT = 0.15               
TEST_SPLIT = 0.15
SEQUENCE_LENGTH = 10             
 
CNN_UNITS = 32   
DENSE_UNITS = 32
DROPOUT_RATE = 0.4       
LEARNING_RATE = 0.0004 
BATCH_SIZE = 64  
EPOCHS = 50                     
PATIENCE_ES = 5               
PATIENCE_LR = 5   

MOVE_THRESHOLD = 0.001 
HORIZON = 5                  
COMMISSION = 0.001 
SLIPPAGE = 0.0005 
MAX_POSITION_SIZE = 0.10 
MAX_DRAWDOWN = 0.15  

MODEL_PATH = '../models/model_cnn'

In [11]:
def create_model(input_shape, name="cnn_model"):
    model = Sequential(name=name)
    model.add(Input(shape=input_shape))
    
    model.add(Conv1D(filters=64, kernel_size=3, activation='relu', padding='same'))
    model.add(BatchNormalization())      
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(DROPOUT_RATE))
    
    model.add(Conv1D(filters=128, kernel_size=3, activation='relu', padding='same'))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(DROPOUT_RATE))
    
    model.add(Flatten())
    model.add(Dense(DENSE_UNITS, activation='relu'))
    model.add(Dropout(DROPOUT_RATE))
    
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

In [12]:
# ЗАГРУЗКА ДАННЫХ
raw_data, raw_df = download(TICKER, START_DATE, END_DATE, interval=INTERVAL)
print(f"📊 Загружено {len(raw_df)} записей")

spy = yf.download("SPY", start=START_DATE, end=END_DATE, progress=False)
vix = yf.download("^VIX", start=START_DATE, end=END_DATE, progress=False)

spy.columns = spy.columns.get_level_values(0)
vix.columns = vix.columns.get_level_values(0)

# ПОДГОТОВКА ДАННЫХ
print("\n🔧 Подготовка данных...")
df = calculate_base_features(raw_df)
df = df.dropna().reset_index(drop=True)

res_prepare_data = prepare_data(
    df, TRAIN_SPLIT, VAL_SPLIT, SEQUENCE_LENGTH,
    move_threshold=MOVE_THRESHOLD, horizon=HORIZON, FEATURES1=FEATURES
)

📊 Загружено 2768 записей

🔧 Подготовка данных...
✅ Создано 34 признаков. Всего строк: 2768
✅ Train: 1924 | Val: 404 | Test: 405


In [13]:
# STAGE 1: MOVE and HOLD
res_stage1 = stage1(
    res_prepare_data['X_train'], res_prepare_data['X_val'], res_prepare_data['X_test'],
    res_prepare_data['y1_train'], res_prepare_data['y1_val'], res_prepare_data['y1_test'], 
    res_prepare_data['cw1'], res_prepare_data['df_test'],
    create_model=create_model,
    optimize_threshold=optimize_threshold,
    filter_signals=filter_signals,
    EPOCHS=EPOCHS, 
    BATCH_SIZE=BATCH_SIZE, 
    PATIENCE_ES=PATIENCE_ES, 
    PATIENCE_LR=PATIENCE_LR
)


 STAGE 1: MOVE vs HOLD
Epoch 1/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.5333 - loss: 0.9624 - val_accuracy: 0.4554 - val_loss: 0.7295 - learning_rate: 4.0000e-04
Epoch 2/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5426 - loss: 0.9078 - val_accuracy: 0.4604 - val_loss: 0.7723 - learning_rate: 4.0000e-04
Epoch 3/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5634 - loss: 0.8346 - val_accuracy: 0.4876 - val_loss: 0.7783 - learning_rate: 4.0000e-04
Epoch 4/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5520 - loss: 0.8151 - val_accuracy: 0.4975 - val_loss: 0.7707 - learning_rate: 4.0000e-04
Epoch 5/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5691 - loss: 0.7878 - val_accuracy: 0.5173 - val_loss: 0.7902 - learning_rate: 4.0000e-04
Epoch 6/50
 1/31 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6875 - loss: 0.6141
Epoch 6: ReduceLROnPlateau reducing learning rate to 0.00019999999494757503.
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accu

In [14]:
# STAGE 2: UP and DOWN
res_stage2 = stage2(
    X_train=res_prepare_data['X_train'],
    X_val=res_prepare_data['X_val'],
    X_test=res_prepare_data['X_test'],
    y2_train=res_prepare_data['y2_train'],
    y2_val=res_prepare_data['y2_val'],
    y2_test=res_prepare_data['y2_test'],
    mask1_train=res_stage1['mask_train'],
    mask1_val=res_stage1['mask_val'],
    mask1_test=res_stage1['mask_test'],
    create_model=create_model,
    optimize_threshold=optimize_threshold,
    EPOCHS=EPOCHS,
    BATCH_SIZE=BATCH_SIZE,
    PATIENCE_ES=PATIENCE_ES,
    PATIENCE_LR=PATIENCE_LR
    #PROB_THRESHOLD_BASE=PROB_THRESHOLD_BASE
)


 STAGE 2: UP vs DOWN
Stage1 signals → train: 1400, val: 53, test: 0
Class weights: {np.int64(0): np.float64(1.2237762237762237), np.int64(1): np.float64(0.8454106280193237)}
Epoch 1/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.5143 - loss: 1.2665 - val_accuracy: 0.2642 - val_loss: 0.7665 - learning_rate: 4.0000e-04
Epoch 2/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5143 - loss: 1.0396 - val_accuracy: 0.2642 - val_loss: 0.7703 - learning_rate: 4.0000e-04
Epoch 3/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5307 - loss: 0.9715 - val_accuracy: 0.2830 - val_loss: 0.7436 - learning_rate: 4.0000e-04
Epoch 4/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5364 - loss: 0.8934 - val_accuracy: 0.5283 - val_loss: 0.7117 - learning_rate: 4.0000e-04
Epoch 5/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5443 - loss: 0.8585 - val_accuracy: 0.6415 - val_loss: 0.6990 - learning_rate: 4.0000e-04
Epoch 6/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - acc

In [15]:
# Оценка 
# Stage1
print_stage_report(
    res_prepare_data['y1_test'], 
    res_stage1['preds_test'], 
    "Stage 1 (MOVE vs HOLD)", 
    ['HOLD', 'MOVE']
)

# Stage2
print_stage_report(
    res_stage2['y_true'], 
    res_stage2['preds'], 
    "Stage 2 (UP vs DOWN)", 
    ['DOWN', 'UP']
)


Stage 1 (MOVE vs HOLD):
              precision    recall  f1-score   support

        HOLD       0.50      1.00      0.66       200
        MOVE       1.00      0.01      0.03       205

    accuracy                           0.50       405
   macro avg       0.75      0.51      0.35       405
weighted avg       0.75      0.50      0.34       405


Stage 2 (UP vs DOWN):
Stage 2 (UP vs DOWN): нет данных для оценки


In [16]:
from sklearn.metrics import roc_auc_score

roc_auc_score(res_prepare_data['y1_test'], res_stage1['probs_test'])
roc_auc_score(res_stage2['y_true'], res_stage2['probs'])
probs_full = res_stage2['model'].predict(res_prepare_data['X_test']).flatten()

results = production_backtest(
    res_prepare_data['df_test'], 
    res_stage1['probs_test'],
    probs_full,
    threshold1=res_stage1['threshold'], 
    threshold2=res_stage2['threshold'],
    commission=COMMISSION, 
    slippage=SLIPPAGE,
    max_position_size=MAX_POSITION_SIZE,
    horizon=HORIZON
)

history_stages = [res_stage1['history'], res_stage2['history']]
thresholds = {"S1": res_stage1['threshold'], "S2": res_stage2['threshold']}
backtest(
    results=results,
    history_stages=history_stages,   
    strategy_name="Two-Stage CNN Strategy",
    thresholds=thresholds,
    save_dir="../models/model_cnn"              
)


ValueError: Found array with 0 sample(s) (shape=(0,)) while a minimum of 1 is required.

In [ ]:
# Сохранение модели, scaler и параметров
params = {
    'threshold_stage1': res_stage1['threshold'],
    'threshold_stage2': res_stage2['threshold'],
    'move_threshold': MOVE_THRESHOLD,
    'horizon': HORIZON,
    'commission': COMMISSION,
    'slippage': SLIPPAGE,
    'max_position_size': MAX_POSITION_SIZE,
    'max_drawdown': MAX_DRAWDOWN,
    'sequence_length': SEQUENCE_LENGTH
}

backtest_results = {
    "total_return": results['total_return'],
    "sharpe_ratio": results['sharpe'],
    "max_drawdown": results['max_drawdown'],
    "win_rate": results['win_rate'],
    "trades": results['trades'],
    "avg_pnl": results['avg_pnl']
}

save_artifacts_json(
    base_dir=MODEL_PATH,
    strategy_name="two_stage",
    res_stage1=res_stage1,
    res_stage2=res_stage2,
    FEATURES=FEATURES,
    params=params,
    backtest_results=backtest_results
)

save_models(
    model_stage1=res_stage1['model'],
    model_stage2=res_stage2['model'],
    scaler=res_prepare_data['scaler'],
    params=params,
    base_dir=MODEL_PATH,
    features=FEATURES
)

generate_final_report(
    results=results,
    model_type="two_stage",
    stage1_metrics={
        "threshold": res_stage1['threshold'],
        "f1": res_stage1['f1_val']
    },
    stage2_metrics={
        "threshold": res_stage2['threshold'],
        "f1": res_stage2['f1_val']
    },
    ml_metrics={
        "Stage 1 F1": res_stage1['f1_val'],
        "Stage 2 F1": res_stage2['f1_val']
    }
)

✅ JSON summary saved: ../models/model_cnn/two_stage/artifacts_summary.json
✅ Model Stage 1 saved: ../models/model_cnn/strategy/model_stage1.h5
✅ Model Stage 2 saved: ../models/model_cnn/strategy/model_stage2.h5
✅ Scaler, features, and params (WITH features list) saved to ../models/model_cnn/strategy

📁 Все артефакты сохранены в ../models/model_cnn/strategy

📋 ИТОГОВЫЙ ОТЧЁТ

🎯 Two-Stage Classification:
   Stage 1: MOVE vs HOLD (Threshold=0.48000000000000004, F1=0.7007)
   Stage 2: UP vs DOWN (Threshold=0.45, F1=0.6049)

Метрики:
   Stage 1 F1: 0.7007
   Stage 2 F1: 0.6049

💰 Production Backtest:
   Total Return: -0.5%
   Sharpe Ratio: -0.28
   Max Drawdown: 1.4%
Trades: 10
   Win Rate: 40.0%
